# Prediksi Harga Komoditas Pangan Strategis

Notebook ini menjalankan alur penelitian dari data gabungan sampai evaluasi LSTM/GRU. Data demo sintetis hanya untuk memeriksa kode; jangan gunakan untuk kesimpulan penelitian.

## 1. Persiapan data
Letakkan `prices.csv` di `data/raw/` dengan kolom `date,commodity,price`; data cuaca dan kalender bersifat opsional. Petunjuk sumber dan format ada di `README.md`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from pangan_forecast.data import load_and_merge
from pangan_forecast.models import run_experiments
from pangan_forecast.recommend import make_recommendation
print('Proyek:', ROOT)

In [ ]:
clean = load_and_merge(ROOT / 'data/raw/prices.csv', ROOT / 'data/raw/weather.csv', ROOT / 'data/raw/calendar.csv')
clean.to_csv(ROOT / 'data/processed/combined_daily.csv', index=False)
clean.head(), clean.groupby('commodity').agg(n=('price','size'), mulai=('date','min'), akhir=('date','max'))

## 2. Eksplorasi ringkas
Periksa rentang tanggal, satuan, lokasi, data hilang, dan lonjakan sebelum pemodelan. Interpolasi pada pipeline hanya mengisi sela di antara observasi.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for commodity, group in clean.groupby('commodity'):
    ax.plot(group['date'], group['price'], label=commodity, alpha=.8)
ax.set(title='Harga harian per komoditas', ylabel='Harga (Rp, satuan sumber)')
ax.legend(ncol=2); ax.grid(alpha=.2); plt.show()

## 3. Latih dan bandingkan model
Split kronologis: 80% periode lebih awal untuk pelatihan dan 20% periode akhir untuk pengujian. Setiap model memakai jendela 30 hari dan menghasilkan 7 langkah. Eksperimen membandingkan LSTM/GRU dengan harga saja dan dengan cuaca/kalender historis.

In [ ]:
metrics, forecasts = run_experiments(clean, ROOT, window=30, horizon=7, epochs=30)
metrics.sort_values(['commodity', 'MAE'])

## 4. Grafik prakiraan dan rekomendasi
Pilih satu komoditas dan titik awal uji untuk membandingkan prediksi dengan harga aktual. Ambang rekomendasi awal ±5% dapat disesuaikan dan harus dijustifikasi di metodologi.

In [ ]:
if not forecasts.empty:
    item = forecasts['commodity'].iloc[0]
    model = forecasts['model'].iloc[0]
    feature_set = forecasts['features'].iloc[0]
    sample = forecasts[(forecasts.commodity == item) & (forecasts.model == model) & (forecasts.features == feature_set)]
    origin = sample['origin_date'].max()
    sample = sample[sample.origin_date == origin].sort_values('step')
    plt.figure(figsize=(9, 4))
    plt.plot(sample.target_date, sample.actual, marker='o', label='Aktual')
    plt.plot(sample.target_date, sample.predicted, marker='o', label='Prediksi')
    plt.legend(); plt.grid(alpha=.2); plt.title(f'{item} — {model}, {feature_set}'); plt.show()
    make_recommendation(sample.reference_price.iloc[0], sample.predicted.tolist())

## 5. Simpan tabel evaluasi
MAE/RMSE dalam satuan harga pada sumber; MAPE dalam persen. Tulis keterbatasan data, prosedur pemilihan wilayah/pasar, dan hasil pengecekan asumsi pada laporan skripsi.

In [ ]:
metrics.to_csv(ROOT / 'reports/model_comparison.csv', index=False)
forecasts.to_csv(ROOT / 'reports/forecasts.csv', index=False)
print('Hasil tersimpan di folder reports')